# **Arquitectura y fuente (Inicio)**

* **DATASET:** 311 Service Requests from 2020 to Present.
* **LINK:** https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9/about_data
* **TAMAÑO:** Cuenta con 23 millones de registros y 44 columnas.
* **COLUMNAS CON FECHA:**
  * **Created Date :** Fecha y hora en que el ciudadano envía la solicitud.
  * **Closed Date :** Fecha y hora en que la agencia cierra la solicitud, es nula mientras la solicitud sigue abierta.
* **COLUMNAS:**
  * **Unique Key :** Identificador único de cada solicitud.
  * **Agency :** Sigla de la agencia municipal responsable (por ejemplo NYPD, DOT, HPD).
  * **Agency Name :** Nombre completo de la agencia.
  * **Problem (formerly Complaint Type) :** Tipo general del problema reportado (por ejemplo ruido, calefacción).
  * **Problem Detail (formerly Descriptor) :** Detalle que precisa el tipo de problema.
  * **Additional Details :** Información extra sobre el problema.
  * **Location Type :** Tipo de lugar donde ocurre el problema.
  * **Incident Zip :** Código postal del lugar del incidente.
  * Incident Address : Dirección del incidente.
  * **Street Name :** Nombre de la calle del incidente.
  * **Cross Street 1 :** Primera calle que cruza cerca del incidente.
  * **Cross Street 2 :** Segunda calle que cruza cerca del incidente.
  * **Intersection Street 1 :** Primera calle que se cruza, basada en la ubicación del incidente validada geográficamente
* **ACTUALIZACIÓN:** Desde 2020 hasta la actualidad.
* **USUARIOS:** Administrador que necesite mirar información importante o relevante sobre las reseñas e identificar cuales serian los procesos a mejorar

La clave o campo que identifica como único cada registro es **Unike key**, esta es asignada a cada queja o reseña.

**Inidicadores** que pueden ser claves:
  * Cuantas solicitudes se hacen por año o mes
  * Cual es el top de queja (Con columna **problem**)
  * Tiempo de tardanza en cerrar una queja o reseña (se puede medir en año o mes)


# **Diagrama estructurado**

In [1]:
# Carga de Librerías
import os
import sys
import re
import json
import glob
import time
import base64
import shutil
import socket
import sqlite3
import tarfile
import zipfile
import hashlib
import requests
import subprocess
import pandas as pd
from datetime import datetime, timedelta
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
from IPython.display import HTML, display

# Librerías de Apache Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

In [2]:
# CONFIGURACIÓN DEL ENTORNO (WINDOWS)

# Carpeta del proyecto: aquí quedan raw/, evidencias/ y la base SQLite que leerá el tablero.
DIR_PROYECTO = os.path.abspath(os.getcwd())

# Carpeta de trabajo de Big Data: Java, Hadoop, los datos de HDFS y las salidas de Spark.
# Va fuera de OneDrive (la sincronización bloquea archivos que Hadoop y Spark renombran)
# y sin espacios en la ruta (los scripts .cmd de Hadoop no los soportan).
DIR_BIGDATA = os.environ.get("BIGDATA_HOME", os.path.join(os.path.expanduser("~"), "bigdata"))
if " " in DIR_BIGDATA:
    raise ValueError(f"La ruta {DIR_BIGDATA} tiene espacios. Define la variable BIGDATA_HOME con una ruta sin espacios, por ejemplo C:\\bigdata")
os.makedirs(DIR_BIGDATA, exist_ok=True)

# Intérprete de Python del entorno virtual (lo usan Spark y los scripts de MapReduce)
PYTHON_EXE = sys.executable


def ruta_spark(ruta):
    """Convierte una ruta local de Windows en una URI que Spark y Hadoop entienden (file:///C:/...)."""
    return "file:///" + os.path.abspath(ruta).replace("\\", "/")


def descargar(url, destino):
    """Descarga un archivo grande mostrando el avance cada 10 %."""
    print(f"[*] Descargando {url}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total, hecho, ultimo = int(r.headers.get("content-length", 0)), 0, -10
        with open(destino + ".part", "wb") as f:
            for bloque in r.iter_content(chunk_size=1024 * 1024):
                f.write(bloque)
                hecho += len(bloque)
                if total and hecho * 100 // total >= ultimo + 10:
                    ultimo = hecho * 100 // total
                    print(f"    {ultimo}% ({hecho / 1024**2:.0f} MB)")
    os.replace(destino + ".part", destino)


print(f"[+] Carpeta del proyecto: {DIR_PROYECTO}")
print(f"[+] Carpeta de Big Data:  {DIR_BIGDATA}")
print(f"[+] Python:               {PYTHON_EXE}")

[+] Carpeta del proyecto: C:\Users\juanl\OneDrive\Documentos\proyecto-final-pipeline-datos
[+] Carpeta de Big Data:  C:\Users\juanl\bigdata
[+] Python:               C:\Users\juanl\OneDrive\Documentos\proyecto-final-pipeline-datos\.venv\Scripts\python.exe


In [3]:

diagrama = """
flowchart LR
    API["API Socrata<br/>NYC 311"] -->|"Python requests<br/>paginación + reintentos"| RAW["RAW<br/>JSONL por fecha_ingesta<br/>+ manifiesto"]
    RAW -->|"Productor Python<br/>key = unique_key"| K["Apache Kafka<br/>tópico, 3 particiones"]
    K -->|"Consumidor con grupo<br/>1 archivo por lote"| H["HDFS + MapReduce<br/>/datalake/fuente/dataset/<br/>fecha_ingesta=..."]
    H -->|"PySpark<br/>limpieza + dedup"| C["CURADA<br/>Parquet particionado<br/>por año"]
    C -->|"Validaciones +<br/>marca de agua"| S["SERVICIO<br/>SQLite: 3 indicadores"]
    S --> D["Tablero<br/>Plotly HTML"]
    K -.->|"Structured Streaming<br/>ventana + watermark + checkpoint"| F["Capa rápida"]
    F -.-> S
"""

b64 = base64.urlsafe_b64encode(diagrama.encode("utf-8")).decode("ascii")
url = f"https://mermaid.ink/img/{b64}?bgColor=FFFFFF"      # fondo, para cambiar el color
r = requests.get(url, timeout=60)
r.raise_for_status()
open("diagrama_pipeline.png", "wb").write(r.content)

display(HTML(f'<img src="{url}" style="max-width:100%">'))

**Descripción**

* **Zona raw:** Se utiliza la API y se guardan los datos en JSONL por fecha de ingesta, junto con un manifiesto. Un productor publica cada registro en Kafka (3 particiones, con unique_key como llave principal), y se guarda en HDFS, un archivo por lote. Con esos datos se compara un conteo hecho en MapReduce contra el mismo cálculo en Spark.
* **Zona curada:** PySpark lee de HDFS, quita duplicados por unique_key, corrige tipos, nulos y formatos, y guarda en Parquet particionado por el nivel de tiempo que se requiera si por año o mes. Luego se aplican las validaciones de calidad, que permiten cargar solo lo nuevo y repetir el proceso sin duplicar registros.
* **Zona de servicio:** los 3 indicadores ya calculados se guardan en SQLite, y el tablero solo los lee, sin hacer cálculos.

Todo esto ayuda a crear una trazabilidad de punta a punta, se puede saber de dónde vino cada registro, cuándo se ingresó y qué le pasó en cada etapa. El sobre JSON guarda la fuente, el dataset, la fecha y el registro original intacto, y el `unique_key` permite seguir una misma solicitud desde la API hasta el tablero.

La arquitectura que se escogió fue **lambda**, que combina dos caminos, la capa batch y la capa streaming y luego se unen en la muestra del tablero, se eligió sobre Kappa porque una queja o reseña que hoy está abierta, mañana se cierra, y entonces cambian su estado y su fecha de cierre. Por eso conviene revisar todo en lotes con limpieza y validaciones, y no solo sumar lo que va llegando como se realiza en Kappa, claramente los dos caminos(batch, streaming) compartirán la misma lógica de código.


## **Pregunta Principal**

Una vez analisado la fuente de datos hemos desarrollado la pregunta principal donde buscamos orientar el proyecto hacia el análisis de la gestión de solicitudes ciudadanas y los problemas urbanos de Nueva York:

**¿Cuáles son los principales problemas urbanos reportados por los ciudadanos de Nueva York y cómo varían los tiempos de atención y resolución según el tipo de problema, la agencia responsable y la ubicación?**

Usuarios de los indicadores:
- Ciudadanos interesados en la gestión de su ciudad.
- Investigadores de gestión urbana.
- Administradores públicos.
- Analistas de servicios ciudadanos.

## **Ingesta a la zona raw**

### **¿Qué buscamos con la Zona Raw y qué contiene?**

El objetivo principal de la **Zona Raw** es actuar como nuestro repositorio histórico de seguridad. Buscamos almacenar los datos tal y como provienen de la fuente original (la API de NYC 311) sin ningún tipo de transformación ni alteración, garantizando que siempre podamos volver a procesarlos si las reglas de negocio cambian.

Esta etapa contiene:
1. **El "sobre" JSON**: Cada registro de la API se envuelve en una estructura que contiene:
   * **fuente**: El origen del dato (Socrata API NYC).
   * **dataset**: El identificador del conjunto de datos.
   * **id_registro**: El identificador único (`unique_key`).
   * **fecha_ingesta**: La fecha exacta en que se descargó el dato (formato `AAAA-MM-DD`).
   * **payload**: El registro original completamente intacto.
2. **Particionado físico**: Los datos se almacenan en archivos JSON Lines (`.jsonl`) organizados en carpetas con la estructura `fecha_ingesta=AAAA-MM-DD`.
3. **Manifiesto de control**: Un archivo `manifiesto.json` que registra el estado de la ejecución (URL, parámetros, número de registros ingestados y ruta del archivo resultante) para asegurar la trazabilidad del pipeline.

In [4]:
# CONFIGURACIÓN Y PARÁMETROS
ENDPOINT_API = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
FUENTE = "Socrata API NYC"
DATASET_ID = "erm2-nwe9"
LIMIT_POR_PAGINA = 1000   # Tamaño del lote por petición
DIAS_HISTORIA = 30        # Ventana de análisis: los últimos 30 días con datos publicados
MAX_REGISTROS = 50000     # Tamaño total de la muestra
REGISTROS_POR_DIA = MAX_REGISTROS // DIAS_HISTORIA   # Cuota diaria: la muestra cubre todo el periodo

# Generamos la fecha de ingesta actual
fecha_hoy = datetime.now().strftime("%Y-%m-%d")

# La API publica con algunas horas de retraso: tomamos como último día el de la solicitud más reciente
ultima_fecha_api = requests.get(ENDPOINT_API, params={"$select": "max(created_date) AS ultima"}, timeout=60).json()[0]["ultima"]
ultimo_dia = datetime.fromisoformat(ultima_fecha_api[:10]).date()
dias_consulta = [ultimo_dia - timedelta(days=i) for i in range(DIAS_HISTORIA)]   # del más reciente al más antiguo
print(f"[+] Periodo a ingerir: {dias_consulta[-1]} a {dias_consulta[0]} ({REGISTROS_POR_DIA} registros por día)")

[+] Periodo a ingerir: 2026-10-25 a 2026-11-23 (1666 registros por día)


In [5]:


# CONFIGURACIÓN DE REINTENTOS CON URLLIB3
def configurar_sesion_con_reintentos():
    sesion = requests.Session()
    # Reintentará ante códigos de error comunes de red o sobrecarga (500, 502, 503, 504 o 429)
    retries = Retry(
        total=5,                  # Número máximo de intentos totales
        backoff_factor=1,         # Factor de espera exponencial (e.g., 1s, 2s, 4s...)
        status_forcelist=[429, 500, 502, 503, 504],
        raise_on_status=False
    )
    adaptador = HTTPAdapter(max_retries=retries)
    sesion.mount("http://", adaptador)
    sesion.mount("https://", adaptador)
    return sesion

In [6]:
# CREACIÓN DE DIRECTORIOS PARA EL DATALAKE RAW
ruta_particion = os.path.join("raw", f"fecha_ingesta={fecha_hoy}")
os.makedirs(ruta_particion, exist_ok=True)
archivo_salida = os.path.join(ruta_particion, "datos.jsonl")

sesion_cliente = configurar_sesion_con_reintentos()
offset = 0
total_ingestados = 0
continuar_paginacion = True

print(f"[*] Iniciando ingesta en la ruta: {ruta_particion}")

[*] Iniciando ingesta en la ruta: raw\fecha_ingesta=2026-11-24


In [7]:
# DESCARGA POR DÍA
# Cada día aporta la misma cuota de registros para que la muestra cubra los 30 días.
# Dentro de cada día se pide en el orden del identificador interno de Socrata (:id), que no depende
# de la hora, así la muestra se reparte a lo largo de todo el día (y no solo en las últimas horas).
# Al escribir se ordena por fecha descendente: el archivo queda con lo más reciente primero.
with open(archivo_salida, "w", encoding="utf-8") as f_out:
    for dia in dias_consulta:
        inicio, fin = dia.isoformat(), (dia + timedelta(days=1)).isoformat()
        registros_dia, offset = [], 0
        while len(registros_dia) < REGISTROS_POR_DIA:
            limite = min(LIMIT_POR_PAGINA, REGISTROS_POR_DIA - len(registros_dia))
            parametros = {
                "$where": f"created_date >= '{inicio}T00:00:00' AND created_date < '{fin}T00:00:00'",
                "$order": ":id",
                "$limit": limite,
                "$offset": offset,
            }
            try:
                respuesta = sesion_cliente.get(ENDPOINT_API, params=parametros, timeout=60)
                respuesta.raise_for_status()
                registros = respuesta.json()
            except Exception as e:
                print(f"    [Err] Ocurrió un error en la llamada de red ({inicio}, offset={offset}): {e}")
                break

            # Si no hay más registros devueltos por la API para ese día, pasamos al siguiente
            if not registros:
                break
            registros_dia.extend(registros)
            offset += len(registros)
            if len(registros) < limite:
                break

        registros_dia.sort(key=lambda r: r.get("created_date", ""), reverse=True)
        for registro in registros_dia:
            # Estructuramos el sobre JSON requerido, con la llave única (unique_key)
            sobre = {
                "fuente": FUENTE,
                "dataset": DATASET_ID,
                "id_registro": registro.get("unique_key", "unknown_id"),
                "fecha_ingesta": fecha_hoy,
                "payload": registro
            }
            # Escribimos como una línea en el archivo JSON Lines
            f_out.write(json.dumps(sobre, ensure_ascii=False) + "\n")
            total_ingestados += 1
        print(f"    [~] {inicio}: {len(registros_dia)} registros")

if total_ingestados == 0:
    raise RuntimeError("La API no devolvió registros. Revisa la conexión a internet.")

    [~] 2026-11-23: 1666 registros


    [~] 2026-11-22: 1666 registros


    [~] 2026-11-21: 1666 registros


    [~] 2026-11-20: 1666 registros


    [~] 2026-11-19: 1666 registros


    [~] 2026-11-18: 1666 registros


... (20 outputs left out)

    [~] 2026-10-28: 1666 registros


    [~] 2026-10-27: 1666 registros


    [~] 2026-10-26: 1666 registros


    [~] 2026-10-25: 1666 registros


In [8]:
# CREACIÓN DEL MANIFIESTO DE CONTROL
manifiesto = {
    "fecha_ejecucion": datetime.now().isoformat(),
    "url_endpoint": ENDPOINT_API,
    "parametros_utilizados": {
        "limite_solicitado_por_pagina": LIMIT_POR_PAGINA,
        "max_registros_permitidos": MAX_REGISTROS,
        "dias_historia": DIAS_HISTORIA,
        "registros_por_dia": REGISTROS_POR_DIA,
        "fecha_inicio": str(dias_consulta[-1]),
        "fecha_fin": str(dias_consulta[0]),
        "orden_dentro_del_dia": ":id (muestra repartida a lo largo del día)"
    },
    "total_registros_descargados": total_ingestados,
    "archivo_destino": archivo_salida
}

ruta_manifiesto = os.path.join(ruta_particion, "manifiesto.json")
with open(ruta_manifiesto, "w", encoding="utf-8") as f_man:
    json.dump(manifiesto, f_man, indent=4, ensure_ascii=False)

print(f"\n[+] Ingesta completada con éxito.")
print(f"[+] Registros almacenados: {total_ingestados}")
print(f"[+] Manifiesto generado en: {ruta_manifiesto}")


[+] Ingesta completada con éxito.
[+] Registros almacenados: 49980
[+] Manifiesto generado en: raw\fecha_ingesta=2026-11-24\manifiesto.json


In [9]:
# VERIFICACIÓN DE LOS ARCHIVOS GENERADOS

# Mostramos el contenido del manifiesto
with open(ruta_manifiesto, "r", encoding="utf-8") as m:
    print("--- CONTENIDO DEL MANIFIESTO ---")
    display(json.load(m))

# Cargamos las primeras filas de la zona raw para comprobar la correcta estructura del Sobre JSON
lineas_lectura = []
with open(archivo_salida, "r", encoding="utf-8") as f:
    for _ in range(3):
        linea = f.readline()
        if linea:
            lineas_lectura.append(json.loads(linea))

print("\n--- MUESTRA DE LOS SOBRES JSON INGESTADOS (Primeros 3 registros) ---")
for idx, item in enumerate(lineas_lectura):
    print(f"\n[Registro {idx+1}]")
    print(json.dumps(item, indent=2, ensure_ascii=False)[:600] + "... [Truncado para vista]")

--- CONTENIDO DEL MANIFIESTO ---


{'fecha_ejecucion': '2026-11-24T19:25:11.391197',
 'url_endpoint': 'https://data.cityofnewyork.us/resource/erm2-nwe9.json',
 'parametros_utilizados': {'limite_solicitado_por_pagina': 1000,
  'max_registros_permitidos': 50000,
  'dias_historia': 30,
  'registros_por_dia': 1666,
  'fecha_inicio': '2026-10-25',
  'fecha_fin': '2026-11-23',
  'orden_dentro_del_dia': ':id (muestra repartida a lo largo del día)'},
 'total_registros_descargados': 49980,
 'archivo_destino': 'raw\\fecha_ingesta=2026-11-24\\datos.jsonl'}


--- MUESTRA DE LOS SOBRES JSON INGESTADOS (Primeros 3 registros) ---

[Registro 1]
{
  "fuente": "Socrata API NYC",
  "dataset": "erm2-nwe9",
  "id_registro": "71227813",
  "fecha_ingesta": "2026-11-24",
  "payload": {
    "unique_key": "71227813",
    "created_date": "2026-11-23T12:06:02.000",
    "agency": "DSNY",
    "agency_name": "Department of Sanitation",
    "complaint_type": "Graffiti",
    "descriptor": "Graffiti",
    "incident_zip": "10471",
    "incident_address": "6425 BROADWAY",
    "street_name": "BROADWAY",
    "address_type": "ADDRESS",
    "city": "BRONX",
    "facility_type": "N/A",
    "status": "Open",
    "due_date": "2026-12-23T12:06:02.000",
    "res... [Truncado para vista]

[Registro 2]
{
  "fuente": "Socrata API NYC",
  "dataset": "erm2-nwe9",
  "id_registro": "71227506",
  "fecha_ingesta": "2026-11-24",
  "payload": {
    "unique_key": "71227506",
    "created_date": "2026-11-23T12:02:40.000",
    "agency": "DSNY",
    "agency_name": "Department of Sanitat

## **Parte 3: Mensajería con Kafka**

### **¿Qué pasa si el productor de Kafka se ejecuta dos veces y cómo evitamos duplicados?**

Si nuestro productor se ejecuta dos veces sin controles adicionales, se enviarán todos los registros de nuevo al tópico de Kafka, duplicando el volumen de datos en el flujo. Esto ocurre porque Kafka, por defecto, se comporta bajo el modelo **At Least Once** (Al menos una vez), priorizando que ningún dato se pierda a costa de poder duplicarlos.

#### **Cómo evitar la duplicidad en el mundo real:**

1. **En el Productor (Idempotencia):**
   * Podemos habilitar la propiedad `enable.idempotence=true` en el cliente productor de Kafka. Al hacer esto, el broker de Kafka asigna un ID único al productor junto con un número de secuencia a cada mensaje. Si el productor reenvía un lote que el broker ya recibió y confirmó con éxito, el broker simplemente lo descarta, evitando duplicar datos en el tópico de forma transparente.

2. **En el Consumidor (Procesamiento Idempotente / Deduplicación):**
   * **Uso del Id de Registro (`unique_key`)**: Dado que empaquetamos cada mensaje con un `id_registro` (la llave única original), el consumidor o las etapas posteriores del flujo (como Spark o la base de datos de servicio) pueden comprobar si ese identificador ya fue procesado antes de guardarlo definitivamente.
   * **Estrategia de Upsert**: Al escribir en el almacenamiento de destino, en lugar de hacer operaciones de inserción directa (*Insert*), se realizan actualizaciones-inserciones (*Upsert*). Si la llave existe, se sobrescribe; si no existe, se crea.

In [10]:
# CLASES DEL SIMULADOR DE KAFKA

class SimuladorKafka:
    """Simula un Broker de Kafka con N particiones para un tópico."""
    def __init__(self, topico, num_particiones=3):
        self.topico = topico
        self.num_particiones = num_particiones
        # Inicializamos las particiones como listas vacías
        self.particiones = {i: [] for i in range(num_particiones)}
        print(f"[Kafka Broker] Tópico '{self.topico}' creado con {self.num_particiones} particiones (0, 1 y 2).")

    def obtener_particion_destinataria(self, llave):
        """Asigna la partición aplicando un hash MD5 sobre la llave del mensaje (unique_key)"""
        if not llave:
            return 0
        hash_llave = hashlib.md5(str(llave).encode('utf-8')).hexdigest()
        # Convertimos el hash a número entero y aplicamos módulo para balancear uniformemente
        return int(hash_llave, 16) % self.num_particiones

    def enviar_mensaje(self, llave, valor):
        particion = self.obtener_particion_destinataria(llave)
        # Cada mensaje en el broker tiene su llave, valor y un offset incremental simulado
        offset_nuevo = len(self.particiones[particion])
        mensaje = {
            "key": llave,
            "value": valor,
            "offset": offset_nuevo
        }
        self.particiones[particion].append(mensaje)
        return particion, offset_nuevo


class ProductorKafka:
    """Simula el comportamiento de un productor de Kafka"""
    def __init__(self, broker):
        self.broker = broker

    def publicar_archivo_raw(self, ruta_jsonl):
        print(f"\n[Productor] Iniciando lectura y publicación de {ruta_jsonl}...")
        mensajes_enviados = 0

        if not os.path.exists(ruta_jsonl):
            print(f"[Productor Err] El archivo {ruta_jsonl} no existe.")
            return

        with open(ruta_jsonl, 'r', encoding='utf-8') as f:
            for linea in f:
                registro_sobre = json.loads(linea.strip())
                # Tomamos como llave de mensaje el 'id_registro' establecido en el sobre JSON
                llave_mensaje = registro_sobre.get("id_registro")

                # Enviamos el registro completo al broker
                particion, offset = self.broker.enviar_mensaje(llave_mensaje, registro_sobre)
                mensajes_enviados += 1

                # Mostramos una muestra pequeña del ruteo inicial para no saturar la pantalla
                if mensajes_enviados <= 5:
                    print(f"  [Productor] Msg {mensajes_enviados} -> Llave: {llave_mensaje} -> Enviado a Partición {particion} (Offset {offset})")

        print(f"[Productor] Publicación finalizada. Total mensajes enviados al Broker: {mensajes_enviados}")


class ConsumidorKafka:
    """Simula el comportamiento de un consumidor leyendo bajo un grupo específico"""
    def __init__(self, broker, grupo_id="grupo-consumidores-nyc"):
        self.broker = broker
        self.grupo_id = grupo_id
        # Mantiene el registro de hasta qué offset ha leído el grupo en cada partición
        self.offsets_guardados = {i: 0 for i in range(broker.num_particiones)}
        print(f"[Consumidor] Inicializado con Grupo ID: '{self.grupo_id}'.")

    def consumir_mensajes(self, limite_por_particion=3):
        """Lee mensajes pendientes simulando el avance de offsets en cada partición"""
        print(f"\n[Consumidor Grupo: {self.grupo_id}] Iniciando lectura de mensajes...")

        for particion_id, mensajes in self.broker.particiones.items():
            offset_inicial = self.offsets_guardados[particion_id]
            mensajes_pendientes = mensajes[offset_inicial:offset_inicial + limite_por_particion]

            print(f"  [-] Leyendo de Partición {particion_id} desde Offset {offset_inicial} (Mensajes en cola: {len(mensajes_pendientes)})")

            for msg in mensajes_pendientes:
                llave = msg["key"]
                payload_original = msg["value"]["payload"]
                # Extraemos algún dato del payload para verificar contenido real
                tipo_queja = payload_original.get("complaint_type", "Desconocido")
                print(f"    [Recibido] Offset {msg['offset']} | Llave: {llave} | Queja: {tipo_queja}")

                # Actualizamos el puntero de lectura
                self.offsets_guardados[particion_id] = msg["offset"] + 1


In [11]:
# INSTANCIACIÓN Y PRUEBA DE LA SIMULACIÓN

# 1. Creamos el Broker con su tópico y 3 particiones
broker_nyc = SimuladorKafka(topico="nyc-311-raw", num_particiones=3)

# 2. Inicializamos Productor y Consumidor
productor = ProductorKafka(broker_nyc)
consumidor = ConsumidorKafka(broker_nyc, grupo_id="grupo-analisis-quejas")

# 3. Publicamos los datos desde nuestro archivo de la Zona Raw creado en la sección anterior
productor.publicar_archivo_raw(archivo_salida)

# 4. Consumimos una pequeña muestra del inicio de cada una de las 3 particiones para validar
consumidor.consumir_mensajes(limite_por_particion=2)

[Kafka Broker] Tópico 'nyc-311-raw' creado con 3 particiones (0, 1 y 2).
[Consumidor] Inicializado con Grupo ID: 'grupo-analisis-quejas'.

[Productor] Iniciando lectura y publicación de raw\fecha_ingesta=2026-11-24\datos.jsonl...
  [Productor] Msg 1 -> Llave: 71227813 -> Enviado a Partición 2 (Offset 0)
  [Productor] Msg 2 -> Llave: 71227506 -> Enviado a Partición 2 (Offset 1)
  [Productor] Msg 3 -> Llave: 71227812 -> Enviado a Partición 2 (Offset 2)
  [Productor] Msg 4 -> Llave: 71227809 -> Enviado a Partición 0 (Offset 0)
  [Productor] Msg 5 -> Llave: 71227808 -> Enviado a Partición 2 (Offset 3)


[Productor] Publicación finalizada. Total mensajes enviados al Broker: 49980

[Consumidor Grupo: grupo-analisis-quejas] Iniciando lectura de mensajes...
  [-] Leyendo de Partición 0 desde Offset 0 (Mensajes en cola: 2)
    [Recibido] Offset 0 | Llave: 71227809 | Queja: Street Condition
    [Recibido] Offset 1 | Llave: 71228479 | Queja: Street Condition
  [-] Leyendo de Partición 1 desde Offset 0 (Mensajes en cola: 2)
    [Recibido] Offset 0 | Llave: 71228559 | Queja: Blocked Driveway
    [Recibido] Offset 1 | Llave: 71228560 | Queja: Illegal Parking
  [-] Leyendo de Partición 2 desde Offset 0 (Mensajes en cola: 2)
    [Recibido] Offset 0 | Llave: 71227813 | Queja: Graffiti
    [Recibido] Offset 1 | Llave: 71227506 | Queja: Graffiti


# **Parte 4 : Data lake en HDFS y MapReduce.**

**¿Qué hacemos aquí?** El consumidor toma los mensajes del tópico y los guarda en HDFS con la estructura `fuente/dataset/fecha_ingesta=AAAA-MM-DD`. Luego se hace el mismo conteo de quejas por tipo con MapReduce y con Spark, para comprobar que dan el mismo resultado.

**¿Por qué un archivo por lote y no por mensaje?** HDFS está pensado para pocos archivos grandes. Millones de archivos pequeños saturan el NameNode (que guarda los metadatos de cada archivo) y hacen lenta cualquier lectura.

**¿Por qué comparar MapReduce con Spark?** Es una validación cruzada: si dos motores distintos dan el mismo conteo, los datos llegaron completos a HDFS. Además muestra la diferencia entre ambos: MapReduce escribe a disco entre etapas y Spark trabaja en memoria, por eso es más rápido.

Se configura y levanta HDFS (1 NameNode + 1 DataNode) de forma nativa en Windows.

* Primer paso: instalar Java 11 (versión portátil de Eclipse Temurin, sin permisos de administrador) y verificar PySpark. Hadoop 3.3.6 solo soporta Java 8 y 11.

In [12]:
# JAVA 11 PORTÁTIL + PYSPARK
DIR_JAVA = os.path.join(DIR_BIGDATA, "jdk-11")
JAVA_EXE = os.path.join(DIR_JAVA, "bin", "java.exe")

if not os.path.exists(JAVA_EXE):
    print("[*] Instalando Java 11 (Eclipse Temurin)...")
    zip_java = os.path.join(DIR_BIGDATA, "jdk11.zip")
    descargar("https://api.adoptium.net/v3/binary/latest/11/ga/windows/x64/jdk/hotspot/normal/eclipse", zip_java)
    tmp_java = os.path.join(DIR_BIGDATA, "jdk_tmp")
    shutil.rmtree(tmp_java, ignore_errors=True)
    with zipfile.ZipFile(zip_java) as z:
        z.extractall(tmp_java)
    # El zip trae una carpeta del tipo jdk-11.0.x+y; la renombramos a jdk-11
    shutil.move(os.path.join(tmp_java, os.listdir(tmp_java)[0]), DIR_JAVA)
    shutil.rmtree(tmp_java)
    os.remove(zip_java)
    print("[+] Java instalado.")
else:
    print("[+] Java ya estaba instalado.")

os.environ["JAVA_HOME"] = DIR_JAVA
print("   ", subprocess.run([JAVA_EXE, "-version"], capture_output=True, text=True).stderr.splitlines()[0])

try:
    import pyspark
    print(f"[+] PySpark ya estaba instalado (versión {pyspark.__version__}).")
except ImportError:
    print("[*] Instalando PySpark 3.5.1...")
    subprocess.check_call([PYTHON_EXE, "-m", "pip", "install", "-q", "pyspark==3.5.1"])
    print("[+] PySpark instalado.")

# PySpark 3.5 usa el módulo 'distutils' (en toPandas), que Python 3.12 ya no incluye; setuptools lo provee
try:
    import setuptools
except ImportError:
    subprocess.check_call([PYTHON_EXE, "-m", "pip", "install", "-q", "setuptools"])
    import setuptools

[+] Java ya estaba instalado.


    openjdk version "11.0.33" 2026-10-20
[+] PySpark ya estaba instalado (versión 3.5.1).


* Se descarga Hadoop

In [13]:
HADOOP_VERSION = "3.3.6"
DIR_HADOOP = os.path.join(DIR_BIGDATA, "hadoop")
tar_hadoop = os.path.join(DIR_BIGDATA, f"hadoop-{HADOOP_VERSION}.tar.gz")

if os.path.isdir(DIR_HADOOP):
    print(f"[+] La carpeta {DIR_HADOOP} ya existe, no hace falta descargar.")
elif os.path.exists(tar_hadoop):
    print("[+] El archivo de Hadoop ya estaba descargado.")
else:
    print(f"[*] Descargando Hadoop {HADOOP_VERSION} (unos 700 MB)...")
    espejos = [
        f"https://dlcdn.apache.org/hadoop/common/hadoop-{HADOOP_VERSION}/hadoop-{HADOOP_VERSION}.tar.gz",
        f"https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VERSION}/hadoop-{HADOOP_VERSION}.tar.gz",
    ]
    for url in espejos:
        try:
            descargar(url, tar_hadoop)
            break
        except Exception as e:
            print(f"    [!] Falló {url}: {e}")
    else:
        raise RuntimeError("No se pudo descargar Hadoop desde ningún espejo.")
    print(f"[+] Descarga finalizada. Tamaño del archivo: {os.path.getsize(tar_hadoop) / 1024**2:.0f} MB")

[+] La carpeta C:\Users\juanl\bigdata\hadoop ya existe, no hace falta descargar.


Se descomprime y limpia. En Windows Hadoop necesita además dos binarios nativos (`winutils.exe` y `hadoop.dll`), que se descargan del repositorio [cdarlint/winutils](https://github.com/cdarlint/winutils). Se omite la carpeta `share/doc` porque solo trae documentación y sus rutas superan el límite de 260 caracteres de Windows.

In [14]:
if not os.path.isdir(DIR_HADOOP):
    print("[*] Descomprimiendo Hadoop...")
    tmp_hadoop = os.path.join(DIR_BIGDATA, "hadoop_tmp")
    shutil.rmtree(tmp_hadoop, ignore_errors=True)
    with tarfile.open(tar_hadoop) as t:
        # Solo archivos y carpetas normales (Windows no crea enlaces simbólicos sin permisos)
        miembros = [m for m in t.getmembers() if (m.isfile() or m.isdir()) and "/share/doc/" not in m.name]
        t.extractall(tmp_hadoop, members=miembros, filter="data")
    shutil.move(os.path.join(tmp_hadoop, f"hadoop-{HADOOP_VERSION}"), DIR_HADOOP)
    shutil.rmtree(tmp_hadoop)
    os.remove(tar_hadoop)
    print(f"[+] Hadoop listo en {DIR_HADOOP}.")
else:
    print("[+] Hadoop ya está descomprimido.")

# Binarios nativos de Hadoop para Windows
for nombre in ["winutils.exe", "hadoop.dll"]:
    destino = os.path.join(DIR_HADOOP, "bin", nombre)
    if not os.path.exists(destino):
        descargar(f"https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-{HADOOP_VERSION}/bin/{nombre}", destino)
print("[+] winutils.exe y hadoop.dll disponibles.")

[+] Hadoop ya está descomprimido.
[+] winutils.exe y hadoop.dll disponibles.


In [15]:
# VARIABLES DE ENTORNO
os.environ["JAVA_HOME"] = DIR_JAVA
os.environ["HADOOP_HOME"] = DIR_HADOOP
os.environ["HADOOP_CONF_DIR"] = os.path.join(DIR_HADOOP, "etc", "hadoop")
os.environ["PYSPARK_PYTHON"] = PYTHON_EXE
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_EXE
os.environ["PYTHONIOENCODING"] = "utf-8"   # los scripts de MapReduce leen y escriben en UTF-8

# Evitar acumular repetidamente las mismas rutas en el PATH
for ruta_bin in [os.path.join(DIR_HADOOP, "bin"), os.path.join(DIR_JAVA, "bin")]:
    if ruta_bin not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] = ruta_bin + os.pathsep + os.environ["PATH"]


def sh(cmd, check=True, solo_stdout=False):
    """Ejecuta un comando en cmd.exe. Con solo_stdout=True se ignoran los avisos que la JVM escribe en stderr."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, encoding="utf-8", errors="replace")
    out = (r.stdout if solo_stdout else r.stdout + r.stderr).strip()
    if check and r.returncode != 0:
        raise RuntimeError(f"Falló: {cmd}\n{(r.stdout + r.stderr).strip()}")
    return out


def uri_local(ruta):
    """Ruta local en formato URI para los archivos de configuración de Hadoop."""
    return "file:///" + ruta.replace("\\", "/")


DIR_HDFS = os.path.join(DIR_BIGDATA, "hdfs")
DIR_LOGS = os.path.join(DIR_BIGDATA, "logs")
DIR_TMP = os.path.join(DIR_BIGDATA, "tmp")
for d in [DIR_HDFS, DIR_LOGS, DIR_TMP]:
    os.makedirs(d, exist_ok=True)

# CONFIGURACIÓN DE HADOOP (todos los servicios escuchan solo en localhost)
# NameNode en 9820 (puerto estándar de Hadoop 3): el 9000 lo suelen ocupar los kernels de Jupyter
PUERTO_NAMENODE = 9820
conf = os.environ["HADOOP_CONF_DIR"]
with open(os.path.join(conf, "core-site.xml"), "w", encoding="utf-8") as f:
    f.write(f"""<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:{PUERTO_NAMENODE}</value></property>
  <property><name>hadoop.tmp.dir</name><value>{DIR_TMP.replace(chr(92), "/")}</value></property>
</configuration>""")
with open(os.path.join(conf, "hdfs-site.xml"), "w", encoding="utf-8") as f:
    f.write(f"""<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.namenode.name.dir</name><value>{uri_local(os.path.join(DIR_HDFS, "name"))}</value></property>
  <property><name>dfs.datanode.data.dir</name><value>{uri_local(os.path.join(DIR_HDFS, "data"))}</value></property>
  <property><name>dfs.permissions.enabled</name><value>false</value></property>
  <property><name>dfs.namenode.http-address</name><value>localhost:9870</value></property>
  <property><name>dfs.datanode.address</name><value>localhost:9866</value></property>
  <property><name>dfs.datanode.http.address</name><value>localhost:9864</value></property>
  <property><name>dfs.datanode.ipc.address</name><value>localhost:9867</value></property>
  <property><name>dfs.datanode.hostname</name><value>localhost</value></property>
  <property><name>dfs.client.use.datanode.hostname</name><value>true</value></property>
</configuration>""")
with open(os.path.join(conf, "mapred-site.xml"), "w", encoding="utf-8") as f:
    f.write("""<configuration>
  <property><name>mapreduce.framework.name</name><value>local</value></property>
</configuration>""")

if not os.path.exists(os.path.join(DIR_HDFS, "name", "current")):
    print("[*] Formateando el NameNode...")
    sh("hdfs namenode -format -force -nonInteractive")


def puerto_abierto(puerto):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("localhost", puerto)) == 0


def proceso_en_puerto(puerto):
    """Devuelve (pid, nombre del ejecutable) del proceso que escucha en el puerto, o (None, None)."""
    for linea in sh("netstat -ano -p tcp", check=False).splitlines():
        partes = linea.split()
        if len(partes) == 5 and partes[3] == "LISTENING" and partes[1].rsplit(":", 1)[-1] == str(puerto):
            info = sh(f'tasklist /FI "PID eq {partes[4]}" /FO CSV /NH', check=False)
            return partes[4], info.split(",")[0].strip('"') if info.startswith('"') else "desconocido"
    return None, None


def iniciar_demonio(nombre, puerto, espera=90):
    """Lanza un servicio de HDFS en segundo plano (equivale a 'hdfs --daemon start')."""
    if puerto_abierto(puerto):
        pid, ejecutable = proceso_en_puerto(puerto)
        if ejecutable and ejecutable.lower() == "java.exe":
            print(f"[+] {nombre} ya estaba en ejecución (puerto {puerto}, PID {pid}).")
            return
        raise RuntimeError(f"El puerto {puerto} que necesita el {nombre} lo está usando otro programa "
                           f"({ejecutable}, PID {pid}). Ciérralo o cambia el puerto en la configuración.")
    log = open(os.path.join(DIR_LOGS, f"{nombre}.log"), "a", encoding="utf-8")
    subprocess.Popen(f"hdfs {nombre}", shell=True, stdout=log, stderr=subprocess.STDOUT,
                     creationflags=subprocess.CREATE_NO_WINDOW)
    for _ in range(espera):
        if puerto_abierto(puerto):
            print(f"[+] {nombre} iniciado (puerto {puerto}).")
            return
        time.sleep(1)
    raise RuntimeError(f"{nombre} no arrancó. Revisa el log: {os.path.join(DIR_LOGS, nombre + '.log')}")


def detener_hdfs():
    """Detiene los procesos que escuchan en los puertos de HDFS (equivale a 'hdfs --daemon stop')."""
    pids = set()
    for linea in sh("netstat -ano -p tcp", check=False).splitlines():
        partes = linea.split()
        if len(partes) == 5 and partes[3] == "LISTENING" and partes[1].rsplit(":", 1)[-1] in {str(PUERTO_NAMENODE), "9866", "9870"}:
            pids.add(partes[4])
    for pid in pids:
        sh(f"taskkill /F /T /PID {pid}", check=False)
    print(f"[+] HDFS detenido ({len(pids)} procesos).")


iniciar_demonio("namenode", PUERTO_NAMENODE)
iniciar_demonio("datanode", 9866)
print(sh("hdfs dfsadmin -safemode wait", solo_stdout=True))
print(sh("hdfs dfs -ls /", check=False, solo_stdout=True))
print("[+] Interfaz web del NameNode: http://localhost:9870")

[+] namenode iniciado (puerto 9820).


[+] datanode iniciado (puerto 9866).


Safe mode is OFF


Found 2 items
drwxr-xr-x   - juanl supergroup          0 2026-11-24 16:32 /datalake
drwxr-xr-x   - juanl supergroup          0 2026-11-24 19:09 /mr_out
[+] Interfaz web del NameNode: http://localhost:9870


consumidor que guarda en HDFS, un archivo por lote

In [16]:

# La ruta de HDFS no debe tener espacios: "Socrata API NYC" -> "socrata_api_nyc"
FUENTE_SLUG = re.sub(r"[^a-z0-9]+", "_", FUENTE.lower()).strip("_")
HDFS_BASE = f"/datalake/{FUENTE_SLUG}/{DATASET_ID}"
HDFS_URI = f"hdfs://localhost:{PUERTO_NAMENODE}"

class ConsumidorHDFS(ConsumidorKafka):
    """Consumidor que guarda lotes de mensajes en HDFS (un archivo por lote)."""

    def consumir_a_hdfs(self, max_por_particion=2000):
        total, nlote = 0, 0
        fechas_reiniciadas = set()   # si se re-ingiere la misma fecha, la partición se reemplaza (idempotencia)
        while True:
            pendientes = []
            for pid, mensajes in self.broker.particiones.items():
                ini = self.offsets_guardados[pid]
                pendientes += [(pid, m) for m in mensajes[ini: ini + max_por_particion]]
            if not pendientes:
                break

            # Agrupamos el lote por fecha_ingesta (una carpeta por fecha)
            por_fecha = {}
            for pid, m in pendientes:
                sobre = m["value"]
                por_fecha.setdefault(sobre["fecha_ingesta"], []).append(json.dumps(sobre, ensure_ascii=False))

            for fecha, lineas in por_fecha.items():
                destino = f"{HDFS_BASE}/fecha_ingesta={fecha}"
                if fecha not in fechas_reiniciadas:
                    sh(f'hdfs dfs -rm -r -f "{destino}"')
                    fechas_reiniciadas.add(fecha)
                sh(f'hdfs dfs -mkdir -p "{destino}"')
                local = os.path.join(DIR_TMP, f"lote_{nlote:05d}.jsonl")
                with open(local, "w", encoding="utf-8") as f:
                    f.write("\n".join(lineas) + "\n")
                sh(f'hdfs dfs -put -f "{local}" "{destino}/lote_{nlote:05d}.jsonl"')
                os.remove(local)

            # Los offsets se confirman DESPUÉS de guardar en HDFS (si falla, no se pierde nada)
            for pid, m in pendientes:
                self.offsets_guardados[pid] = max(self.offsets_guardados[pid], m["offset"] + 1)

            total += len(pendientes)
            nlote += 1
            print(f"  [Consumidor] Lote {nlote}: {len(pendientes)} mensajes guardados en HDFS")
        return total, nlote

# Grupo nuevo: lee el tópico completo desde el offset 0
consumidor_hdfs = ConsumidorHDFS(broker_nyc, grupo_id="grupo-hdfs")
total_hdfs, lotes_hdfs = consumidor_hdfs.consumir_a_hdfs()
print(f"\nMensajes guardados en HDFS: {total_hdfs} en {lotes_hdfs} lotes")

[Consumidor] Inicializado con Grupo ID: 'grupo-hdfs'.


  [Consumidor] Lote 1: 6000 mensajes guardados en HDFS


  [Consumidor] Lote 2: 6000 mensajes guardados en HDFS


  [Consumidor] Lote 3: 6000 mensajes guardados en HDFS


  [Consumidor] Lote 4: 6000 mensajes guardados en HDFS


  [Consumidor] Lote 5: 6000 mensajes guardados en HDFS


  [Consumidor] Lote 6: 6000 mensajes guardados en HDFS


  [Consumidor] Lote 7: 6000 mensajes guardados en HDFS


  [Consumidor] Lote 8: 6000 mensajes guardados en HDFS


  [Consumidor] Lote 9: 1980 mensajes guardados en HDFS

Mensajes guardados en HDFS: 49980 en 9 lotes


Evidencia del data lake

In [17]:
EVID = os.path.join(DIR_PROYECTO, "evidencias")
os.makedirs(EVID, exist_ok=True)

listado = sh(f'hdfs dfs -ls -R "{HDFS_BASE}"', solo_stdout=True)
# Contamos en Python solo las líneas de datos (stdout), sin los avisos de la JVM
contenido_hdfs = sh(f'hdfs dfs -cat "{HDFS_BASE}/*/*.jsonl"', solo_stdout=True)
lineas_hdfs = sum(1 for linea in contenido_hdfs.splitlines() if linea.strip())
reporte = (f"{listado}\n\nLíneas en HDFS: {lineas_hdfs} | "
           f"Registros descargados de la API: {total_ingestados} | "
           f"¿Coinciden?: {lineas_hdfs == total_ingestados}")
with open(os.path.join(EVID, "04_hdfs_ls.txt"), "w", encoding="utf-8") as f:
    f.write(reporte)
print(reporte)

drwxr-xr-x   - juanl supergroup          0 2026-11-24 19:27 /datalake/socrata_api_nyc/erm2-nwe9/fecha_ingesta=2026-11-24
-rw-r--r--   1 juanl supergroup   10034083 2026-11-24 19:26 /datalake/socrata_api_nyc/erm2-nwe9/fecha_ingesta=2026-11-24/lote_00000.jsonl
-rw-r--r--   1 juanl supergroup    9829095 2026-11-24 19:26 /datalake/socrata_api_nyc/erm2-nwe9/fecha_ingesta=2026-11-24/lote_00001.jsonl
-rw-r--r--   1 juanl supergroup    9956489 2026-11-24 19:26 /datalake/socrata_api_nyc/erm2-nwe9/fecha_ingesta=2026-11-24/lote_00002.jsonl
-rw-r--r--   1 juanl supergroup    9699999 2026-11-24 19:26 /datalake/socrata_api_nyc/erm2-nwe9/fecha_ingesta=2026-11-24/lote_00003.jsonl
-rw-r--r--   1 juanl supergroup   10142808 2026-11-24 19:27 /datalake/socrata_api_nyc/erm2-nwe9/fecha_ingesta=2026-11-24/lote_00004.jsonl
-rw-r--r--   1 juanl supergroup    9818146 2026-11-24 19:27 /datalake/socrata_api_nyc/erm2-nwe9/fecha_ingesta=2026-11-24/lote_00005.jsonl
-rw-r--r--   1 juanl supergroup   10666511 2026-11-

En la versión de Colab el conteo se hacía con `hdfs dfs -cat ... | wc -l` y se sumaban 2 líneas de advertencia de la máquina virtual (`Cgroup memory/cpu controller...`), por lo que daba 5002 en vez de 5000.

En esta versión para Windows el conteo se hace en Python leyendo solo la salida estándar (`stdout`) del comando, así que los avisos de la JVM (que van a `stderr`) ya no se cuentan y el total coincide exactamente con los registros descargados de la API.

Mapper y reducer

In [18]:
mapper = '''import sys, json
sys.stdin.reconfigure(encoding="utf-8", errors="replace")
sys.stdout.reconfigure(encoding="utf-8", newline="\\n")
for linea in sys.stdin:
    try:
        sobre = json.loads(linea)
        clave = str(sobre["payload"].get("complaint_type", "SIN_DATO")).strip()
        print(clave + "\\t1")
    except Exception:
        pass
'''
reducer = '''import sys
sys.stdin.reconfigure(encoding="utf-8", errors="replace")
sys.stdout.reconfigure(encoding="utf-8", newline="\\n")
actual, suma = None, 0
for linea in sys.stdin:
    linea = linea.rstrip("\\r\\n")
    if "\\t" not in linea:
        continue
    clave, v = linea.split("\\t", 1)
    if clave != actual:
        if actual is not None:
            print(actual + "\\t" + str(suma))
        actual, suma = clave, 0
    suma += int(v)
if actual is not None:
    print(actual + "\\t" + str(suma))
'''
DIR_MR = os.path.join(DIR_BIGDATA, "mapreduce")
os.makedirs(DIR_MR, exist_ok=True)
MAPPER = os.path.join(DIR_MR, "mapper.py")
REDUCER = os.path.join(DIR_MR, "reducer.py")
with open(MAPPER, "w", encoding="utf-8") as f:
    f.write(mapper)
with open(REDUCER, "w", encoding="utf-8") as f:
    f.write(reducer)


def ejecutar_script(script, entrada):
    r = subprocess.run([PYTHON_EXE, script], input=entrada, capture_output=True, text=True, encoding="utf-8", check=True)
    return r.stdout


# Prueba local rápida (sin Hadoop) para validar la lógica con 200 líneas: head | mapper | sort | reducer
with open(archivo_salida, encoding="utf-8") as f:
    muestra = "".join(f.readline() for _ in range(200))
mapeado = ejecutar_script(MAPPER, muestra)
ordenado = "".join(sorted(mapeado.splitlines(keepends=True)))
print("\n".join(ejecutar_script(REDUCER, ordenado).splitlines()[:10]))

Abandoned Vehicle	8
Air Quality	2
Animal in a Park	1
Animal-Abuse	1
Bike Rack	1
Blocked Driveway	17
Consumer Complaint	1
Curb Condition	1
Damaged Tree	4
Day Care	1


Correr MapReduce en Hadoop

In [19]:
jar = glob.glob(os.path.join(DIR_HADOOP, "share", "hadoop", "tools", "lib", "hadoop-streaming-*.jar"))[0]
sh("hdfs dfs -rm -r -f /mr_out/conteo_quejas")
sh("hdfs dfs -mkdir -p /mr_out")

# En modo local los scripts se ejecutan en esta misma máquina, así que se indican con su ruta completa
salida = sh(f'hadoop jar "{jar}" '
            f'-mapper "{PYTHON_EXE} {MAPPER}" -reducer "{PYTHON_EXE} {REDUCER}" '
            f'-input "{HDFS_BASE}/fecha_ingesta=*/*.jsonl" -output /mr_out/conteo_quejas')
print(salida[-500:])

mr_txt = sh('hdfs dfs -cat "/mr_out/conteo_quejas/part-*"', solo_stdout=True)
resultado_mr = {}
for l in mr_txt.splitlines():
    if "\t" in l:
        clave, valor = l.rstrip("\r").split("\t", 1)
        resultado_mr[clave] = int(valor)
print("Tipos de queja (MapReduce):", len(resultado_mr), "| Total:", sum(resultado_mr.values()))

put records=49980
		Reduce output records=152
		Spilled Records=99960
		Shuffled Maps =9
		Failed Shuffles=0
		Merged Map outputs=9
		GC time elapsed (ms)=77
		Total committed heap usage (bytes)=3313500160
	Shuffle Errors
		BAD_ID=0
		CONNECTION=0
		IO_ERROR=0
		WRONG_LENGTH=0
		WRONG_MAP=0
		WRONG_REDUCE=0
	File Input Format Counters 
		Bytes Read=83567533
	File Output Format Counters 
		Bytes Written=3224
2026-11-24 19:28:38,172 INFO streaming.StreamJob: Output directory: /mr_out/conteo_quejas


Tipos de queja (MapReduce): 152 | Total: 49980


Cálculo en Spark y comparación

In [20]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]").appName("fase4")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "16")   # 200 por defecto: demasiadas para un solo equipo
         .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

# recursiveFileLookup evita que Spark trate 'fecha_ingesta=...' como columna duplicada (ya viene en el sobre)
raw = spark.read.option("recursiveFileLookup", "true").json(f"{HDFS_URI}{HDFS_BASE}")
filas_spark = (raw.select(F.coalesce(F.trim(F.col("payload.complaint_type")), F.lit("SIN_DATO")).alias("tipo"))
                  .groupBy("tipo").count().collect())
resultado_spark = {r["tipo"]: r["count"] for r in filas_spark}

coinciden = resultado_mr == resultado_spark
reporte = (f"Tipos de queja -> MapReduce: {len(resultado_mr)} | Spark: {len(resultado_spark)}\n"
           f"Total -> MapReduce: {sum(resultado_mr.values())} | Spark: {sum(resultado_spark.values())}\n"
           f"¿Los resultados coinciden?: {coinciden}")
with open(os.path.join(EVID, "04_mapreduce_vs_spark.txt"), "w", encoding="utf-8") as f:
    f.write(reporte)
print(reporte)

# Top 5 para mostrar en la sustentación
for tipo, n in sorted(resultado_mr.items(), key=lambda x: -x[1])[:5]:
    print(f"{tipo:<40} {n}")

Tipos de queja -> MapReduce: 152 | Spark: 152
Total -> MapReduce: 49980 | Spark: 49980
¿Los resultados coinciden?: True
Illegal Parking                          8995
Noise - Residential                      7803
HEAT/HOT WATER                           4299
Noise - Street/Sidewalk                  2840
Blocked Driveway                         2562


# **Parte 5 : Zona curada con PySpark.**




### **¿Qué buscamos con la Zona Curada y qué decisiones tomamos?**

La **Zona Curada (o Silver)** tiene como objetivo transformar los datos semiestructurados y crudos de la zona Raw en un conjunto de datos limpio, estandarizado, estructurado de manera eficiente para consultas analíticas y libre de duplicados.

A continuación, se detallan las decisiones técnicas de diseño y negocio implementadas a través de **PySpark**:



#### **1. Deduplicación por `id_registro` (Unique Key)**
* **Decisión:** Eliminamos duplicados basados estrictamente en el campo clave `id_registro` (que hereda el `unique_key` de la API de NYC 311).
* **Por qué:** En sistemas distribuidos y pipelines basados en mensajería (como Kafka), la retransmisión de datos o la ejecución concurrente del productor puede generar registros idénticos.


#### **2. Estandarización y Corrección de Tipos de Datos**
* **Campos Temporales (`created_date` y `closed_date`):**
  * *Conversión:* Pasamos de cadenas de texto (`string`) en formato de tiempo (`timestamp`).
  * *Por qué:* Permite realizar cálculos matemáticos de diferencias de tiempo (duración del caso) y optimiza las funciones analíticas de fecha de Spark.
* **Códigos Postales (`incident_zip`):**
  * *Tratamiento:* Aplicamos una expresión regular (`regexp_extract`) para aislar los primeros 5 dígitos numéricos limpios y evitar cadenas con formatos inconsistentes (por ejemplo, sufijos de 4 dígitos o caracteres extraños).
  * *Por qué:* Los códigos postales en EE.UU. a veces se registran de forma descuidada. Al estandarizarlo a 5 dígitos, facilitamos análisis geoespaciales consistentes.
* **Normalización de Categorías (`complaint_type`):**
  * *Tratamiento:* Eliminamos espacios en blanco redundantes en los extremos (`trim`) y reemplazamos los valores nulos con la constante `"SIN_DATO"`.
  * *Por qué:* Evita que quejas semánticamente idénticas (ej. `" Noise"` vs `"Noise"`) se agrupen por separado en los reportes.


#### **3. Control de Valores Nulos (`Na.fill`)**
* **Decisión:** Rellenamos los campos `borough` con la etiqueta de control `"Unspecified"` y `incident_address` con `"No especificada"` en lugar de dejarlos como nulos nativos.
* **Por qué:* Evitamos errores de puntero nulo (`NullPointerException`) y simplificamos los filtros aplicados en el frontend o tablero visualizador, ofreciendo una categoría limpia para agrupar las filas sin datos geográficos precisos.


#### **4. Estrategia de Almacenamiento y Particionado**
* **Formato:** Almacenamos el resultado en formato columnar **Apache Parquet**.
* **Partición elegida:** Particionado físicamente en carpetas utilizando la columna analítica calculada **`year_created`** (año de creación de la queja).

#### **5. Eficiencia en Almacenamiento (JSONL vs Parquet)**
Al comparar el tamaño del JSON de texto plano en HDFS contra el Parquet estructurado en la zona de almacenamiento:
* **Compresión física:** Parquet codifica cada columna de forma independiente aplicando compresión integrada (Snappy) y codificación por diccionario para valores repetidos (como nombres de agencias o comunas).
* **Reducción de espacio:** El factor de reducción de tamaño suele superar el **80% - 90%**, lo que reduce drásticamente los costos de infraestructura en la nube o almacenamiento local y acelera el tiempo de transferencia de red durante consultas concurrentes.


In [21]:
#Cargar datos de HDFS con PySpark
df_raw = spark.read.option("recursiveFileLookup", "true").json(f"{HDFS_URI}{HDFS_BASE}")
filas_antes = df_raw.count()

# Desempaquetar los campos de la carga útil (payload)
df_desempaquetado = df_raw.select(
    F.col("id_registro"),
    F.col("fecha_ingesta"),
    F.col("payload.created_date").alias("created_date_raw"),
    F.col("payload.closed_date").alias("closed_date_raw"),
    F.col("payload.agency").alias("agency"),
    F.col("payload.agency_name").alias("agency_name"),
    F.col("payload.complaint_type").alias("complaint_type"),
    F.col("payload.descriptor").alias("descriptor"),
    F.col("payload.incident_zip").alias("incident_zip_raw"),
    F.col("payload.incident_address").alias("incident_address"),
    F.col("payload.borough").alias("borough")
)

df_dedup = df_desempaquetado.dropDuplicates(["id_registro"])

#Limpieza y Conversión de tipos de datos:

df_curado = df_dedup.withColumn(
    "created_date", F.to_timestamp(F.col("created_date_raw"), "yyyy-MM-dd'T'HH:mm:ss.SSS")
).withColumn(
    "closed_date", F.to_timestamp(F.col("closed_date_raw"), "yyyy-MM-dd'T'HH:mm:ss.SSS")
).withColumn(
    "incident_zip", F.coalesce(F.regexp_extract(F.col("incident_zip_raw"), r"^(\d{5})", 1), F.lit("Desconocido"))
).withColumn(
    "complaint_type", F.coalesce(F.trim(F.col("complaint_type")), F.lit("SIN_DATO"))
).withColumn(
    "year_created", F.year(F.col("created_date"))
).drop("created_date_raw", "closed_date_raw", "incident_zip_raw")

# Rellenar con nulos de control o valores por defecto para evitar inconsistencias
df_curado = df_curado.na.fill({"borough": "Unspecified", "incident_address": "No especificada"})

filas_despues = df_curado.count()

print(f"=== MÉTRICAS DE CALIDAD ===")
print(f"Filas iniciales en Zona Raw: {filas_antes}")
print(f"Filas en la Zona Curada: {filas_despues}")
print(f"Duplicados eliminados: {filas_antes - filas_despues}")

# Guardar en Parquet particionado por Año

ruta_parquet = os.path.join(DIR_BIGDATA, "curada_parquet")
shutil.rmtree(ruta_parquet, ignore_errors=True)
df_curado.write.mode("overwrite").partitionBy("year_created").parquet(ruta_spark(ruta_parquet))

# Comparación de tamaños de almacenamiento (Solución robusta ante warnings de JVM)
salida_hdfs = sh(f'hdfs dfs -du -s "{HDFS_BASE}"', solo_stdout=True)
# Filtramos líneas que no sean advertencias de Cgroup/JVM
lineas_filtradas = [linea for linea in salida_hdfs.splitlines() if not "[warning]" in linea and linea.strip()]
tam_bytes = int(lineas_filtradas[0].split()[0]) if lineas_filtradas else 0
tam_jsonl = tam_bytes / (1024 * 1024)

tam_parquet = sum(os.path.getsize(os.path.join(dirpath, f)) for dirpath, _, filenames in os.walk(ruta_parquet) for f in filenames) / (1024 * 1024)

print(f"\n=== COMPARACIÓN DE TAMAÑOS ===")
print(f"Tamaño del set JSONL original en HDFS: {tam_jsonl:.3f} MB")
print(f"Tamaño del set Parquet particionado local: {tam_parquet:.3f} MB")
print(f"Factor de reducción de espacio: {(1 - (tam_parquet / tam_jsonl)) * 100:.1f}%")

=== MÉTRICAS DE CALIDAD ===
Filas iniciales en Zona Raw: 49980
Filas en la Zona Curada: 49980
Duplicados eliminados: 0



=== COMPARACIÓN DE TAMAÑOS ===
Tamaño del set JSONL original en HDFS: 79.696 MB
Tamaño del set Parquet particionado local: 1.749 MB
Factor de reducción de espacio: 97.8%


Al analizar los resultados de la ejecución de la Fase 5 con PySpark, podemos extraer las siguientes conclusiones técnicas y de negocio:


#### **1. Calidad de Datos y Deduplicación**
* **Métrica:**
  * Filas iniciales (Zona Raw): **49.980** (30 días × 1.666 registros)
  * Filas finales (Zona Curada): **49.980**
  * Duplicados eliminados: **0**
* **Conclusión:**
  La muestra ingerida directamente desde la API oficial de NYC 311 no contiene registros duplicados nativos bajo el mismo identificador (`unique_key`). Esto demuestra que la fuente origen mantiene una alta consistencia transaccional en sus publicaciones. Sin embargo, contar con la etapa de deduplicación distribuida (`dropDuplicates`) asegura la robustez del pipeline e inmuniza el flujo ante fallos de reintento de red en el productor de Kafka.


#### **2. Eficiencia Extrema de Almacenamiento (JSONL vs Parquet)**
* **Métrica:**
  * Tamaño de JSONL original en HDFS: **≈ 80 MB**
  * Tamaño de Parquet particionado: **≈ 1,7 MB**
  * Factor de reducción de espacio: **≈ 98 %**
  * (Las cifras exactas de cada ejecución se imprimen en la celda anterior y quedan en la tabla `ind_pipeline`.)
* **Conclusión:**
  * **Compresión de metadatos y almacenamiento columnar:** El formato JSONL almacena repetidamente los nombres de los esquemas (las claves de cada campo JSON) en formato de texto plano para cada uno de los casi 50.000 registros. Apache Parquet elimina esta redundancia guardando el esquema una sola vez en el metadato del archivo.
  * **Compresión integrada (Snappy):** Al guardar los datos organizados por columnas en lugar de por filas, los valores de una misma columna (como `agency_name` que repite constantemente 'New York City Police Department') se agrupan juntos. Esto permite que algoritmos como la codificación por diccionario reduzcan el tamaño físico de los archivos casi a cero en lotes pequeños.
  * **Particionado:** La creación de la partición física por año de creación (`year_created`) facilita un acceso veloz y estructurado en disco, permitiendo que futuras consultas analíticas filtren carpetas enteras sin tener que procesar datos de años no solicitados.

# **Parte 6 : Calidad, carga incremental e idempotencia.**

* Validaciones importantes:
  * Verificar que el lote contenga los registros.
  * Se verifica que no hayan duplicados en 'id_registro'.
  * Se valida que la fecha de cierre no se cronologiamente anterior o este adelantada a la fecha de creación.
  
* Carga icremental:
  * Se almacena la última fecha procesa en un archivo marca de agua, mientras que spark filtra los registros mayores a dicha marca.

* Idempotencia
  * Si una queja o reseña abierta se cerro hoy, el registro se actualiza en ves de duplicarse.


In [22]:
RUTA_PARQUET = os.path.join(DIR_BIGDATA, "curada_parquet")
RUTA_WATERMARK = os.path.join(EVID, "watermark.json")
RUTA_REPORTE_CALIDAD = os.path.join(EVID, "06_reporte_calidad.json")
DB_SERVICIO = os.path.join(DIR_PROYECTO, "servicio_nyc311.db")

#Marca de Agua (Watermark)
def leer_watermark():
    if os.path.exists(RUTA_WATERMARK):
        with open(RUTA_WATERMARK, "r", encoding="utf-8") as f:
            return json.load(f).get("ultima_fecha_created", "1900-01-01 00:00:00")
    return "1900-01-01 00:00:00"

def guardar_watermark(nueva_fecha):
    with open(RUTA_WATERMARK, "w", encoding="utf-8") as f:
        json.dump({
            "ultima_fecha_created": str(nueva_fecha),
            "actualizado_en": datetime.now().isoformat()
        }, f, indent=4)

#Inicialización de Base de Datos SQLite (Idempotente)
def inicializar_sqlite():
    conn = sqlite3.connect(DB_SERVICIO)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS solicitudes_311 (
            id_registro TEXT PRIMARY KEY,
            fecha_ingesta TEXT,
            created_date TEXT,
            closed_date TEXT,
            agency TEXT,
            agency_name TEXT,
            complaint_type TEXT,
            descriptor TEXT,
            incident_zip TEXT,
            incident_address TEXT,
            borough TEXT,
            year_created INTEGER,
            tiempo_resolucion_horas REAL
        )
    """)
    conn.commit()
    conn.close()

#Función Principal del Pipeline
def ejecutar_pipeline_incremental(forzar_watermark=None):
    inicializar_sqlite()
    watermark_actual = forzar_watermark if forzar_watermark else leer_watermark()
    print(f"[*] Iniciando ejecución incremental | Marca de agua actual: {watermark_actual}")

    # Leer datos de la Zona Curada
    df_curada = spark.read.parquet(ruta_spark(RUTA_PARQUET))

    # Filtrar solo lo nuevo usando la marca de agua
    df_incremental = df_curada.filter(F.col("created_date") > F.lit(watermark_actual))
    total_leidos = df_incremental.count()


    #Primera Validación
    if df_curada.count() == 0:
        raise ValueError("[ERROR CRÍTICO] La Zona Curada está vacía. Pipeline detenido.")

    if total_leidos == 0:
        print("[!] No hay registros nuevos posteriores a la marca de agua. Nada que procesar.")
        reporte = {
            "fecha_ejecucion": datetime.now().isoformat(),
            "watermark_usado": watermark_actual,
            "registros_leidos": 0,
            "registros_descartados": 0,
            "registros_cargados": 0,
            "estado": "SIN_CAMBIOS_NUEVOS"
        }
        with open(RUTA_REPORTE_CALIDAD, "w", encoding="utf-8") as f:
            json.dump(reporte, f, indent=4)
        return reporte


    #Segunda Validación
    nulos_id = df_incremental.filter(F.col("id_registro").isNull() | (F.trim(F.col("id_registro")) == "")).count()
    duplicados_id = total_leidos - df_incremental.select("id_registro").distinct().count()

    if nulos_id > 0 or duplicados_id > 0:
        raise ValueError(f"[ERROR CRÍTICO] Falló integridad de llave primaria: {nulos_id} nulos, {duplicados_id} duplicados.")


    #Tercera Validación
    nulos_clave = df_incremental.filter(
        F.col("created_date").isNull() | F.col("complaint_type").isNull() | F.col("agency").isNull()
    ).count()

    if nulos_clave > 0:
        raise ValueError(f"[ERROR CRÍTICO] Se encontraron {nulos_clave} registros con nulos en created_date, complaint_type o agency.")


    # Validacion de rango de fechas
    df_validos = df_incremental.filter(
        F.col("closed_date").isNull() | (F.col("closed_date") >= F.col("created_date"))
    )
    registros_validos = df_validos.count()
    registros_descartados = total_leidos - registros_validos

    # Calcular tiempo
    df_final = df_validos.withColumn(
        "tiempo_resolucion_horas",
        F.when(
            F.col("closed_date").isNotNull(),
            F.round((F.unix_timestamp("closed_date") - F.unix_timestamp("created_date")) / 3600.0, 2)
        ).otherwise(None)
    )

    # Convertir fechas a string para SQLite
    df_para_sqlite = df_final.withColumn("created_date", F.col("created_date").cast("string")) \
                             .withColumn("closed_date", F.col("closed_date").cast("string"))

    filas = [tuple(row) for row in df_para_sqlite.select(
        "id_registro", "fecha_ingesta", "created_date", "closed_date",
        "agency", "agency_name", "complaint_type", "descriptor",
        "incident_zip", "incident_address", "borough", "year_created",
        "tiempo_resolucion_horas"
    ).collect()]

    # Carga Idempotente (UPSERT) en SQLite
    conn = sqlite3.connect(DB_SERVICIO)
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT INTO solicitudes_311 (
            id_registro, fecha_ingesta, created_date, closed_date,
            agency, agency_name, complaint_type, descriptor,
            incident_zip, incident_address, borough, year_created,
            tiempo_resolucion_horas
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(id_registro) DO UPDATE SET
            closed_date = excluded.closed_date,
            fecha_ingesta = excluded.fecha_ingesta,
            tiempo_resolucion_horas = excluded.tiempo_resolucion_horas
    """, filas)
    conn.commit()
    conn.close()

    # Actualizar la marca de agua con la fecha máxima cargada
    max_fecha = df_validos.agg(F.max("created_date")).collect()[0][0]
    guardar_watermark(max_fecha)

    # Guardar reporte de calidad
    reporte = {
        "fecha_ejecucion": datetime.now().isoformat(),
        "watermark_anterior": watermark_actual,
        "watermark_nuevo": str(max_fecha),
        "validaciones_criticas": "APROBADAS (Conteo > 0, PK única/sin nulos, Columnas clave sin nulos)",
        "registros_evaluados": total_leidos,
        "registros_descartados_por_rango": registros_descartados,
        "registros_cargados_upsert": len(filas),
        "estado": "EXITOSO"
    }
    with open(RUTA_REPORTE_CALIDAD, "w", encoding="utf-8") as f:
        json.dump(reporte, f, indent=4)

    print(f"[+] Carga completada: {len(filas)} registros procesados | {registros_descartados} descartados.")
    print(f"[+] Nueva marca de agua guardada: {max_fecha}")
    return reporte

In [23]:
reporte = ejecutar_pipeline_incremental()
print(reporte)

[*] Iniciando ejecución incremental | Marca de agua actual: 1900-01-01 00:00:00


[+] Carga completada: 49944 registros procesados | 36 descartados.
[+] Nueva marca de agua guardada: 2026-11-23 12:06:02
{'fecha_ejecucion': '2026-11-24T19:29:29.867988', 'watermark_anterior': '1900-01-01 00:00:00', 'watermark_nuevo': '2026-11-23 12:06:02', 'validaciones_criticas': 'APROBADAS (Conteo > 0, PK única/sin nulos, Columnas clave sin nulos)', 'registros_evaluados': 49980, 'registros_descartados_por_rango': 36, 'registros_cargados_upsert': 49944, 'estado': 'EXITOSO'}


## Streaming: capa rápida o Kappa.


# **Parte 7 : Capa Rápida con Spark Structured Streaming (Arquitectura Lambda)**

En esta sección implementamos el procesamiento en tiempo real utilizando **Spark Structured Streaming**. Simularemos el arribo continuo de datos leyendo desde un directorio de streaming (`streaming_input`), procesaremos las ventanas temporales con un **Watermark** de tolerancia para datos tardíos, y salvaremos el estado con un **Checkpoint** persistente.

### **¿Cómo se integra con la Capa Batch (Arquitectura Lambda)?**
* **Capa Batch:** Procesa lotes completos cada 24 horas (como vimos en las fases anteriores), calcula los 3 indicadores históricos de forma exacta mediante PySpark y actualiza SQLite.
* **Capa Rápida:** Procesa los últimos minutos u horas del tráfico de solicitudes en tiempo real. Almacena ventanas cortas (ej. por hora) en la base de datos de servicio.
* **Unificación en la Capa de Servicio (Servicing Layer):** El tablero de control (Plotly/Dash) realiza una consulta unificada (`UNION` o suma lógica): suma los totales consolidados históricos de la Capa Batch más las métricas en tiempo real que aún no han sido procesadas por el lote nocturno, garantizando una vista de negocio de baja latencia sin duplicados.


In [24]:
import shutil
from pyspark.sql.types import StructType, StructField, StringType

# 1. Definimos rutas locales para la simulación del Stream
STREAM_INPUT_DIR = os.path.join(DIR_BIGDATA, "streaming_input")
STREAM_CHECKPOINT_DIR = os.path.join(DIR_BIGDATA, "streaming_checkpoint")
STREAM_OUTPUT_DIR = os.path.join(DIR_BIGDATA, "streaming_output")

# Limpiamos ejecuciones anteriores para asegurar idempotencia en la demostración
for ruta in [STREAM_INPUT_DIR, STREAM_CHECKPOINT_DIR, STREAM_OUTPUT_DIR]:
    if os.path.exists(ruta):
        shutil.rmtree(ruta)
    os.makedirs(ruta, exist_ok=True)

# 2. Definimos el esquema para estructurar la lectura directa del Sobre JSON
esquema_sobre = StructType([
    StructField("id_registro", StringType(), True),
    StructField("fecha_ingesta", StringType(), True),
    StructField("payload", StructType([
        StructField("created_date", StringType(), True),
        StructField("agency", StringType(), True),
        StructField("complaint_type", StringType(), True)
    ]), True)
])

# Tabla de la capa rápida en la base de servicio (se reinicia para la demostración)
with sqlite3.connect(DB_SERVICIO) as conn:
    conn.execute("DROP TABLE IF EXISTS ventanas_tiempo_real")
    conn.execute("""
        CREATE TABLE ventanas_tiempo_real (
            inicio_ventana TEXT,
            fin_ventana TEXT,
            complaint_type TEXT,
            total_solicitudes INTEGER,
            id_micro_lote INTEGER,
            PRIMARY KEY (inicio_ventana, complaint_type)
        )
    """)

print("[+] Rutas inicializadas para Spark Structured Streaming.")

[+] Rutas inicializadas para Spark Structured Streaming.


In [25]:
# 3. Iniciamos el Stream de Lectura (Read Stream)

# Nos aseguramos de que el directorio exista antes de que Spark intente leer para evitar el error PATH_NOT_FOUND
import os
os.makedirs(STREAM_INPUT_DIR, exist_ok=True)

In [26]:
# Spark monitoreará la carpeta STREAM_INPUT_DIR de forma continua para procesar nuevos JSONL
df_stream_raw = spark.readStream.schema(esquema_sobre).json(ruta_spark(STREAM_INPUT_DIR))

# Desempaquetamos y parseamos tipos temporales para aplicar el Watermark
df_stream_clean = df_stream_raw.select(
    F.col("id_registro"),
    F.to_timestamp(F.col("payload.created_date"), "yyyy-MM-dd'T'HH:mm:ss.SSS").alias("event_time"),
    F.coalesce(F.col("payload.complaint_type"), F.lit("SIN_DATO")).alias("complaint_type")
).filter(F.col("event_time").isNotNull())

# 4. Agregación por ventana con Watermark
# Definimos un Watermark de 1 hora para tolerar quejas retrasadas.
df_ventana_agregada = df_stream_clean \
    .withWatermark("event_time", "1 hour") \
    .groupBy(
        F.window(F.col("event_time"), "15 minutes", "15 minutes"),
        F.col("complaint_type")
    ) \
    .count() \
    .select(
        F.col("window.start").cast("string").alias("inicio_ventana"),
        F.col("window.end").cast("string").alias("fin_ventana"),
        F.col("complaint_type"),
        F.col("count").alias("total_solicitudes")
    )

print("[+] Definición lógica del Stream de Agregación completada.")

[+] Definición lógica del Stream de Agregación completada.

In [27]:
# Verificar el contenido físico del directorio del Stream
import os

contenido = os.listdir(STREAM_INPUT_DIR)
print(f"[*] Ruta monitoreada: {STREAM_INPUT_DIR}")
print(f"[*] Número de archivos detectados: {len(contenido)}")
print("[*] Archivos en el directorio:", contenido)

[*] Ruta monitoreada: C:\Users\juanl\bigdata\streaming_input
[*] Número de archivos detectados: 0
[*] Archivos en el directorio: []


In [28]:
# 5. Iniciamos la escritura continua (Write Stream) en modo 'update'
# Cada micro-lote se guarda en SQLite (capa de servicio) con un UPSERT por ventana y tipo de queja.
# El checkpoint garantiza que el motor sepa en qué offset y estado se quedó tras un reinicio.
# Nota: el sink "memory" no sirve aquí porque en modo update no admite recuperación desde checkpoint.

def guardar_ventanas_sqlite(df_micro_lote, id_micro_lote):
    filas = [tuple(r) + (id_micro_lote,) for r in df_micro_lote.collect()]
    if not filas:
        return
    with sqlite3.connect(DB_SERVICIO) as conn:
        conn.executemany("""
            INSERT INTO ventanas_tiempo_real
                (inicio_ventana, fin_ventana, complaint_type, total_solicitudes, id_micro_lote)
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT(inicio_ventana, complaint_type) DO UPDATE SET
                total_solicitudes = excluded.total_solicitudes,
                id_micro_lote = excluded.id_micro_lote
        """, filas)


def consultar_capa_rapida(limite=5):
    with sqlite3.connect(DB_SERVICIO) as conn:
        top = pd.read_sql(f"SELECT * FROM ventanas_tiempo_real ORDER BY total_solicitudes DESC LIMIT {limite}", conn)
        total = conn.execute("SELECT COALESCE(SUM(total_solicitudes), 0) FROM ventanas_tiempo_real").fetchone()[0]
    return top, total


query_stream = df_ventana_agregada.writeStream \
    .foreachBatch(guardar_ventanas_sqlite) \
    .outputMode("update") \
    .option("checkpointLocation", ruta_spark(STREAM_CHECKPOINT_DIR)) \
    .start()

print(f"[+] Stream iniciado. Estado de la Query: {query_stream.status['message']}")

[+] Stream iniciado. Estado de la Query: Initializing sources


In [29]:
# 6. Simulación del arribo de mensajes (Lote 1)
import os
import time

# Ruta por defecto que definimos en la fase 2 de ingesta raw
# Archivo de la ingesta raw de esta ejecución; si no existe, el más reciente disponible
archivo_salida_recuperado = archivo_salida
if not os.path.exists(archivo_salida_recuperado):
    archivos = sorted(glob.glob(os.path.join("raw", "fecha_ingesta=*", "datos.jsonl")))
    if archivos:
        archivo_salida_recuperado = archivos[-1]

print(f"[*] Leyendo registros desde: {archivo_salida_recuperado}")
with open(archivo_salida_recuperado, "r", encoding="utf-8") as f_in:
    bloque = [f_in.readline() for _ in range(1000)]
# La API entrega primero lo más reciente. Para simular la llegada en orden cronológico,
# el lote 1 son las 500 solicitudes más antiguas del bloque y el lote 2 (siguiente celda) las 500 más recientes.
primeras_lineas = bloque[500:]

lote_1_ruta = os.path.join(STREAM_INPUT_DIR, "lote_001.jsonl")
with open(lote_1_ruta, "w", encoding="utf-8") as f_out:
    f_out.writelines(primeras_lineas)

print("[*] Lote 1 depositado en la carpeta de streaming. Esperando procesamiento...")
query_stream.processAllAvailable()  # Espera a que Spark procese el micro-lote

# Consultamos la tabla de la capa rápida en SQLite
resultado_tiempo_real, total_lote_1 = consultar_capa_rapida()
display(resultado_tiempo_real)
print(f"[+] Solicitudes acumuladas en las ventanas tras el lote 1: {total_lote_1}")

[*] Leyendo registros desde: raw\fecha_ingesta=2026-11-24\datos.jsonl
[*] Lote 1 depositado en la carpeta de streaming. Esperando procesamiento...


,inicio_ventana,fin_ventana,complaint_type,total_solicitudes,id_micro_lote
0,2026-11-23 08:15:00,2026-11-23 08:30:00,Illegal Parking,25,0
1,2026-11-23 08:45:00,2026-11-23 09:00:00,Illegal Parking,22,0
2,2026-11-23 08:30:00,2026-11-23 08:45:00,Illegal Parking,21,0
3,2026-11-23 07:30:00,2026-11-23 07:45:00,Illegal Parking,19,0
4,2026-11-23 07:15:00,2026-11-23 07:30:00,Illegal Parking,17,0


[+] Solicitudes acumuladas en las ventanas tras el lote 1: 500


In [30]:
# 7. Demostración de Idempotencia tras Reinicio y Recuperación con Checkpoint
import os
import time

print("[*] Simulando caída/detención del Stream...")
try:
    query_stream.stop()
except NameError:
    print("[!] query_stream no estaba definido de forma local, procediendo sin detener.")

# Archivo de la ingesta raw de esta ejecución; si no existe, el más reciente disponible
archivo_salida_recuperado = archivo_salida
if not os.path.exists(archivo_salida_recuperado):
    archivos = sorted(glob.glob(os.path.join("raw", "fecha_ingesta=*", "datos.jsonl")))
    if archivos:
        archivo_salida_recuperado = archivos[-1]

# Simulamos la llegada de un Lote 2 mientras el motor de streaming estaba apagado
with open(archivo_salida_recuperado, "r", encoding="utf-8") as f_in:
    # El lote 2 son las 500 solicitudes más recientes (posteriores al lote 1), así la marca de agua no las descarta
    siguientes_lineas = [f_in.readline() for _ in range(500)]

lote_2_ruta = os.path.join(STREAM_INPUT_DIR, "lote_002.jsonl")
with open(lote_2_ruta, "w", encoding="utf-8") as f_out:
    f_out.writelines(siguientes_lineas)

print("[*] Lote 2 depositado. Reiniciando el stream utilizando el mismo Checkpoint...")

# Volvemos a levantar el stream apuntando al mismo directorio de Checkpoint
query_stream_recuperada = df_ventana_agregada.writeStream \
    .foreachBatch(guardar_ventanas_sqlite) \
    .outputMode("update") \
    .option("checkpointLocation", ruta_spark(STREAM_CHECKPOINT_DIR)) \
    .start()

query_stream_recuperada.processAllAvailable()  # Espera el reprocesamiento

# Validamos que los datos se acumularon correctamente y continuó la suma sin duplicaciones ni pérdidas
resultado_recuperado, total_lotes = consultar_capa_rapida()
display(resultado_recuperado)
print(f"[+] Solicitudes acumuladas tras el reinicio: {total_lotes} (lote 1: {total_lote_1} + lote 2: {total_lotes - total_lote_1})")

# Detenemos finalmente el stream
query_stream_recuperada.stop()
print("[+] Prueba de Spark Structured Streaming finalizada de manera exitosa.")

[*] Simulando caída/detención del Stream...
[*] Lote 2 depositado. Reiniciando el stream utilizando el mismo Checkpoint...


,inicio_ventana,fin_ventana,complaint_type,total_solicitudes,id_micro_lote
0,2026-11-23 08:15:00,2026-11-23 08:30:00,Illegal Parking,25,0
1,2026-11-23 08:45:00,2026-11-23 09:00:00,Illegal Parking,22,0
2,2026-11-23 08:30:00,2026-11-23 08:45:00,Illegal Parking,21,0
3,2026-11-23 07:30:00,2026-11-23 07:45:00,Illegal Parking,19,0
4,2026-11-23 09:15:00,2026-11-23 09:30:00,Illegal Parking,18,2


[+] Solicitudes acumuladas tras el reinicio: 1000 (lote 1: 500 + lote 2: 500)
[+] Prueba de Spark Structured Streaming finalizada de manera exitosa.


# **Parte 8 : Zona de servicio, indicadores para el tablero.**

Los indicadores que responden la pregunta principal se calculan **aquí, en el pipeline**, y se guardan como tablas en SQLite (`servicio_nyc311.db`). El tablero (`dashboard/app.py`) solo hace `SELECT` sobre estas tablas, sin agregaciones ni cálculos.

| Tabla | Responde | Contenido |
|---|---|---|
| `ind_problemas_principales` | ¿Cuáles son los principales problemas? | Volumen, % del total, % acumulado, % cerradas y mediana de resolución por tipo de problema |
| `ind_tiempo_por_agencia` | ¿Cómo varían los tiempos según la agencia? | Mediana, promedio y percentil 90 de horas hasta el cierre por agencia |
| `ind_distrito_problema` | ¿Cómo varían según la ubicación? | Volumen, mediana de resolución y % cerradas por distrito y tipo de problema (los 10 principales) |
| `ind_distritos` | Ubicación (resumen) | Los mismos indicadores por distrito |
| `ind_capa_rapida` | Capa rápida (lambda) | Solicitudes por ventana de 15 minutos procesadas por Spark Structured Streaming |
| `ind_pipeline` | Trazabilidad | Evidencias de cada etapa del pipeline (conteos, coincidencias, tamaños, calidad) |
| `ind_metadatos` | Título, fuente y fecha de corte | Fuente, periodo, fecha de corte y el titular que responde la pregunta |

**Decisiones:**
* Los tiempos se calculan solo con solicitudes **cerradas** (`tiempo_resolucion_horas` no nulo). Se usa la **mediana** como medida principal porque los tiempos tienen colas muy largas (unos pocos casos tardan semanas) y el promedio se distorsiona.
* En la comparación por agencia solo se muestran las que tienen al menos 30 solicitudes cerradas, para que la mediana sea estable.
* La fuente de los indicadores es la tabla `solicitudes_311`, que ya pasó las validaciones de calidad y la carga idempotente de la Parte 6.

In [31]:
# PARTE 8: INDICADORES DE LA ZONA DE SERVICIO
MIN_CERRADAS_AGENCIA = 30
TOP_PROBLEMAS_MAPA = 10

with sqlite3.connect(DB_SERVICIO) as conn:
    df_servicio = pd.read_sql("SELECT * FROM solicitudes_311", conn)
    df_ventanas = pd.read_sql("SELECT * FROM ventanas_tiempo_real", conn)

df_servicio["cerrada"] = df_servicio["tiempo_resolucion_horas"].notna()
total_servicio = len(df_servicio)


def calcular_indicadores(df, claves):
    """Volumen, cierre y tiempos de resolución por las columnas indicadas."""
    salida = df.groupby(claves).agg(
        solicitudes=("id_registro", "size"),
        cerradas=("cerrada", "sum"),
        mediana_horas=("tiempo_resolucion_horas", "median"),
        promedio_horas=("tiempo_resolucion_horas", "mean"),
        p90_horas=("tiempo_resolucion_horas", lambda s: s.quantile(0.9)),
    ).reset_index()
    salida["cerradas"] = salida["cerradas"].astype(int)
    salida["pct_total"] = (salida["solicitudes"] / len(df) * 100).round(2)
    salida["pct_cerradas"] = (salida["cerradas"] / salida["solicitudes"] * 100).round(1)
    for col in ["mediana_horas", "promedio_horas", "p90_horas"]:
        salida[col] = salida[col].round(1)
    return salida


def formato_duracion(horas):
    """Texto legible para el tablero: horas si es menos de 2 días, si no, días."""
    if pd.isna(horas):
        return "sin cierres"
    texto = f"{horas:.1f} h" if horas < 48 else f"{horas / 24:.1f} días"
    return texto.replace(".", ",")


# Indicador 1: principales problemas
ind_problemas = calcular_indicadores(df_servicio, ["complaint_type"]).sort_values("solicitudes", ascending=False)
ind_problemas.insert(0, "ranking", range(1, len(ind_problemas) + 1))
ind_problemas["pct_acumulado"] = ind_problemas["pct_total"].cumsum().round(1)

# Indicador 2: tiempos por agencia
ind_agencias = calcular_indicadores(df_servicio, ["agency", "agency_name"])
ind_agencias["muestra_suficiente"] = (ind_agencias["cerradas"] >= MIN_CERRADAS_AGENCIA).astype(int)
ind_agencias["mediana_texto"] = ind_agencias["mediana_horas"].map(formato_duracion)
ind_agencias = ind_agencias.sort_values("mediana_horas")

# Indicador 3: ubicación (distrito x problema, para los 10 problemas principales)
top_tipos = ind_problemas["complaint_type"].head(TOP_PROBLEMAS_MAPA).tolist()
df_con_distrito = df_servicio[df_servicio["borough"] != "Unspecified"]
ind_distrito_problema = calcular_indicadores(df_con_distrito[df_con_distrito["complaint_type"].isin(top_tipos)],
                                             ["borough", "complaint_type"])
ind_distrito_problema["orden_problema"] = ind_distrito_problema["complaint_type"].map({t: i for i, t in enumerate(top_tipos, 1)})
ind_distritos = calcular_indicadores(df_con_distrito, ["borough"]).sort_values("solicitudes", ascending=False)
ind_distritos["mediana_texto"] = ind_distritos["mediana_horas"].map(formato_duracion)

# Capa rápida: total por ventana de 15 minutos y problema más frecuente de cada ventana
if len(df_ventanas):
    principal = (df_ventanas.sort_values("total_solicitudes", ascending=False)
                 .drop_duplicates("inicio_ventana")[["inicio_ventana", "complaint_type"]]
                 .rename(columns={"complaint_type": "problema_principal"}))
    ind_capa_rapida = (df_ventanas.groupby(["inicio_ventana", "fin_ventana"], as_index=False)["total_solicitudes"].sum()
                       .merge(principal, on="inicio_ventana").sort_values("inicio_ventana"))
else:
    ind_capa_rapida = pd.DataFrame(columns=["inicio_ventana", "fin_ventana", "total_solicitudes", "problema_principal"])

# Metadatos: fuente, fecha de corte y titular que responde la pregunta
top3 = ind_problemas.head(3)
pct_top3 = round(top3["pct_total"].sum(), 1)
agencias_validas = ind_agencias[ind_agencias["muestra_suficiente"] == 1]
rapida, lenta = agencias_validas.iloc[0], agencias_validas.iloc[-1]
distrito_lento = ind_distritos.sort_values("mediana_horas").iloc[-1]
distrito_rapido = ind_distritos.sort_values("mediana_horas").iloc[0]
mediana_global = df_servicio["tiempo_resolucion_horas"].median()

# El titular compara los tiempos de los 3 problemas principales (responde "según el tipo de problema")
top3_rapido = top3.sort_values("mediana_horas").iloc[0]
top3_lento = top3.sort_values("mediana_horas").iloc[-1]
titular = (f"{top3.iloc[0]['complaint_type']}, {top3.iloc[1]['complaint_type']} y {top3.iloc[2]['complaint_type']} "
           f"concentran el {str(pct_top3).replace('.', ',')}% de las quejas en Nueva York; resolverlas toma desde "
           f"{formato_duracion(top3_rapido['mediana_horas'])} ({top3_rapido['complaint_type']}) hasta "
           f"{formato_duracion(top3_lento['mediana_horas'])} ({top3_lento['complaint_type']})")

metadatos = {
    "titular": titular,
    "pregunta": ("¿Cuáles son los principales problemas urbanos reportados por los ciudadanos de Nueva York y cómo varían "
                 "los tiempos de atención y resolución según el tipo de problema, la agencia responsable y la ubicación?"),
    "fuente": "NYC Open Data, 311 Service Requests from 2020 to Present",
    "dataset": DATASET_ID,
    "url_fuente": "https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9",
    "fecha_inicio": df_servicio["created_date"].min()[:10],
    "fecha_corte": df_servicio["created_date"].max()[:16],
    "fecha_ingesta": fecha_hoy,
    "total_solicitudes": f"{total_servicio:,}".replace(",", "."),
    "pct_cerradas": f"{df_servicio['cerrada'].mean() * 100:.1f}%".replace(".", ","),
    "mediana_global": formato_duracion(mediana_global),
    "num_tipos": str(df_servicio["complaint_type"].nunique()),
    "num_agencias": str(df_servicio["agency"].nunique()),
    "pct_top3": f"{pct_top3}%".replace(".", ","),
    "agencia_rapida": f"{rapida['agency']} ({formato_duracion(rapida['mediana_horas'])})",
    "agencia_lenta": f"{lenta['agency']} ({formato_duracion(lenta['mediana_horas'])})",
    "distrito_rapido": f"{distrito_rapido['borough']} ({distrito_rapido['mediana_texto']})",
    "distrito_lento": f"{distrito_lento['borough']} ({distrito_lento['mediana_texto']})",
    "min_cerradas_agencia": str(MIN_CERRADAS_AGENCIA),
    "generado_en": datetime.now().strftime("%Y-%m-%d %H:%M"),
}
ind_metadatos = pd.DataFrame(list(metadatos.items()), columns=["clave", "valor"])

# Evidencias de cada etapa del pipeline (trazabilidad para el tablero)
with open(RUTA_REPORTE_CALIDAD, encoding="utf-8") as f:
    reporte_calidad = json.load(f)
evidencias = [
    (1, "Ingesta raw", "Registros descargados de la API", f"{total_ingestados:,}".replace(",", "."), f"{DIAS_HISTORIA} días, {REGISTROS_POR_DIA} por día, JSONL + manifiesto"),
    (2, "Kafka (simulado)", "Mensajes publicados en 3 particiones", f"{sum(len(m) for m in broker_nyc.particiones.values()):,}".replace(",", "."), "Llave = unique_key (hash MD5)"),
    (3, "HDFS", "Líneas guardadas en HDFS", f"{lineas_hdfs:,}".replace(",", "."), f"{lotes_hdfs} lotes; coinciden con la API: {'Sí' if lineas_hdfs == total_ingestados else 'No'}"),
    (4, "MapReduce vs Spark", "Conteo por tipo de queja", f"{len(resultado_mr)} tipos", f"Total MapReduce {sum(resultado_mr.values()):,} vs Spark {sum(resultado_spark.values()):,}; coinciden: {'Sí' if resultado_mr == resultado_spark else 'No'}".replace(",", ".")),
    (5, "Zona curada", "Filas tras deduplicar", f"{filas_despues:,}".replace(",", "."), f"{filas_antes - filas_despues} duplicados eliminados; JSONL {tam_jsonl:.1f} MB a Parquet {tam_parquet:.2f} MB ({(1 - tam_parquet / tam_jsonl) * 100:.1f}% menos)".replace(".", ",")),
    (6, "Calidad e incremental", "Estado de la carga", reporte_calidad.get("estado", "-"), f"Cargados (upsert): {reporte_calidad.get('registros_cargados_upsert', 0):,}".replace(",", ".") + f"; descartados por rango de fechas: {reporte_calidad.get('registros_descartados_por_rango', 0)}; marca de agua: {leer_watermark()}"),
    (7, "Capa rápida", "Solicitudes en ventanas de 15 min", f"{int(df_ventanas['total_solicitudes'].sum()):,}".replace(",", "."), "Spark Structured Streaming con marca de agua de 1 h y checkpoint; reinicio sin duplicados"),
    (8, "Zona de servicio", "Tablas de indicadores en SQLite", "7", "El tablero solo lee estas tablas"),
]
ind_pipeline = pd.DataFrame(evidencias, columns=["orden", "etapa", "indicador", "valor", "detalle"])

tablas = {
    "ind_problemas_principales": ind_problemas,
    "ind_tiempo_por_agencia": ind_agencias,
    "ind_distrito_problema": ind_distrito_problema,
    "ind_distritos": ind_distritos,
    "ind_capa_rapida": ind_capa_rapida,
    "ind_pipeline": ind_pipeline,
    "ind_metadatos": ind_metadatos,
}
with sqlite3.connect(DB_SERVICIO) as conn:
    for nombre, tabla in tablas.items():
        tabla.to_sql(nombre, conn, if_exists="replace", index=False)   # se recalculan completas: idempotente

print("[+] Tablas de indicadores guardadas en", DB_SERVICIO)
for nombre, tabla in tablas.items():
    print(f"    {nombre:<28} {len(tabla):>5} filas")
print("\nTITULAR:", titular)
display(ind_problemas.head(10))

[+] Tablas de indicadores guardadas en C:\Users\juanl\OneDrive\Documentos\proyecto-final-pipeline-datos\servicio_nyc311.db
    ind_problemas_principales      152 filas
    ind_tiempo_por_agencia          14 filas
    ind_distrito_problema           50 filas
    ind_distritos                    5 filas
    ind_capa_rapida                 22 filas
    ind_pipeline                     8 filas
    ind_metadatos                   20 filas

TITULAR: Illegal Parking, Noise - Residential y HEAT/HOT WATER concentran el 42,2% de las quejas en Nueva York; resolverlas toma desde 1,2 h (Noise - Residential) hasta 42,3 h (HEAT/HOT WATER)


,ranking,complaint_type,solicitudes,cerradas,mediana_horas,promedio_horas,p90_horas,pct_total,pct_cerradas,pct_acumulado
71,1,Illegal Parking,8995,8958,1.4,3.1,7.2,18.01,99.6,18.0
96,2,Noise - Residential,7801,7792,1.2,3.9,7.6,15.62,99.9,33.6
60,3,HEAT/HOT WATER,4299,3696,42.3,48.7,100.2,8.61,86.0,42.2
97,4,Noise - Street/Sidewalk,2839,2838,1.0,3.0,5.5,5.68,100.0,47.9
13,5,Blocked Driveway,2562,2558,1.5,3.0,7.4,5.13,99.8,53.0
92,6,Noise - Commercial,1833,1833,0.9,3.4,5.9,3.67,100.0,56.7
137,7,UNSANITARY CONDITION,1522,657,150.2,179.8,371.4,3.05,43.2,59.8
91,8,Noise,1365,1045,47.7,75.6,169.3,2.73,76.6,62.5
124,9,Street Condition,910,750,25.1,63.0,191.4,1.82,82.4,64.3
106,10,PLUMBING,857,454,95.9,134.0,319.0,1.72,53.0,66.0


### **Tablero**

Con las tablas listas, el tablero se ejecuta desde una terminal en la carpeta del proyecto:

```
.venv\Scripts\python dashboard\app.py
```

y se abre en http://127.0.0.1:8050. Lee únicamente `servicio_nyc311.db`, así que no necesita que HDFS ni Spark estén encendidos.

## **Cierre del entorno**

Se detienen Spark y los servicios de HDFS para liberar memoria y puertos. Los datos quedan guardados en disco: al volver a ejecutar el cuaderno, HDFS arranca con el contenido que ya tenía.

In [32]:
for q in spark.streams.active:
    q.stop()
spark.stop()
detener_hdfs()

[+] HDFS detenido (2 procesos).
